# Volume 6, Chapter 4: Compactness (Python companion)

Exercises first, solutions at the end. Compactness on a computer: neural networks as universal approximators, Bernstein's probabilistic polynomials, the iteration that approximates $|t|$, Lebesgue's space-filling curve, Euler polygons that pick different solutions of the same equation, and a fixed point reached with no rate. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P4.1 (★★) CS application: neural networks as universal approximators.** (a) Show that a one-hidden-layer ReLU network $x\mapsto c_0+\sum_jc_j\max(x-b_j,0)$ with knots $b_j=\frac jN$ can reproduce exactly the piecewise linear interpolant of $f$ at the knots; compute the weights for $f(x)=\sin(2\pi x)+\frac12\sin(6\pi x)$ and plot the sup error against $N$. (b) Train a one-hidden-layer $\tanh$ network by gradient descent on the same $f$ for widths $2,4,8,16,32$ and compare the errors. What does density (the Remark after Corollary 4.20) promise, and what does it not?

**P4.2 (★) Bernstein polynomials.** Compute $B_nf$ (Exercise 4.6) for $f(x)=|x-\frac12|$ and for $f(x)=\sin(2\pi x)$ with $n=10,100,1000$. Measure the sup error and guess the rates.

**P4.3 (★) The square-root iteration.** Iterate $p_{n+1}=p_n+\frac12(t^2-p_n^2)$ of Lemma 4.18 on a grid of $[-1,1]$ and compare $\max_t(|t|-p_n(t))$ with the bound $\frac2{en}$. Where on $[-1,1]$ is the error largest?

**P4.4 (★★) Lebesgue's space-filling curve.** Implement the curve $\Psi$ of Exercise 4.7 with $20$ ternary digits, plot its image on $[0,1]$ and the trace of $\Psi$ restricted to $[0,1]$ at increasing resolutions. Check numerically that $\Psi$ is not Lipschitz but is Hölder: estimate $\alpha$ with $|\Psi(x)-\Psi(y)|\le C|x-y|^\alpha$.

**P4.5 (★★) Peano without uniqueness.** Run Euler polygons for $y'=\sqrt{\min(|y|,1)}$ on $[0,1]$ with $y_0=0$ and with $y_0=10^{-12}$. Which solution of Exercise 4.8(c) does each one find? Then solve $y'=3y^{2/3}$, $y(0)=0$, and plot the whole family of solutions $y=\max(t-a,0)^3$.

**P4.6 (★) Edelstein's slow convergence.** Iterate $\sin$ from $x_0=1$ (Exercise 4.4) for $10^6$ steps and check that $x_n\sqrt{n/3}\to1$. How many iterations does Banach's bound promise for a contraction of ratio $k=\sin1$, and why does it not apply?

## Solutions

### P4.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f = lambda x: np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x)
xs = np.linspace(0, 1, 4001)
relu = lambda t: np.maximum(t, 0)

def relu_interpolant(N):
    b = np.arange(N) / N                       # knots 0, 1/N, ..., (N-1)/N
    y = f(np.arange(N + 1) / N)
    slopes = np.diff(y) * N                    # slope on each piece
    c = np.concatenate([[slopes[0]], np.diff(slopes)])   # change of slope at each knot
    return lambda x: y[0] + relu(x[:, None] - b[None, :]) @ c

for N in [4, 8, 16, 32, 64, 128]:
    err = np.max(np.abs(relu_interpolant(N)(xs) - f(xs)))
    print(f"N={N:4d} neurons: sup error {err:.2e}   error*N^2 = {err*N*N:.1f}")

In [ ]:
rng = np.random.default_rng(0)
xt = np.linspace(0, 1, 400)[:, None]; yt = f(xt)

def train(width, steps=20000, lr=1e-2):
    W = rng.normal(0, 4, (1, width)); b = rng.normal(0, 2, width); c = rng.normal(0, 0.1, (width, 1)); c0 = 0.0
    params = [W, b, c]
    m = [np.zeros_like(p) for p in params]; v = [np.zeros_like(p) for p in params]
    for t in range(1, steps + 1):
        h = np.tanh(xt @ W + b); out = h @ c + c0; r = out - yt
        gc = h.T @ r / len(xt); gc0 = r.mean()
        gh = (r @ c.T) * (1 - h ** 2)
        gW = xt.T @ gh / len(xt); gb = gh.mean(0)
        for i, g in enumerate([gW, gb, gc]):     # Adam
            m[i] = 0.9 * m[i] + 0.1 * g; v[i] = 0.999 * v[i] + 0.001 * g * g
            params[i] -= lr * (m[i] / (1 - 0.9 ** t)) / (np.sqrt(v[i] / (1 - 0.999 ** t)) + 1e-8)
        c0 -= lr * gc0
    pred = lambda x: np.tanh(x[:, None] @ W + b) @ c + c0
    return pred

plt.figure(figsize=(7, 3)); plt.plot(xs, f(xs), "k", lw=2, label="f")
for width in [2, 4, 8, 16, 32]:
    pred = train(width)
    err = np.max(np.abs(pred(xs)[:, 0] - f(xs)))
    print(f"width {width:2d}: sup error after training {err:.3f}")
    if width in (4, 16): plt.plot(xs, pred(xs), label=f"tanh net, width {width}")
plt.legend(); plt.show()

(a) A ReLU unit $\max(x-b,0)$ is a hinge: the sum $c_0+\sum_jc_j\max(x-b_j,0)$ is piecewise linear with a change of slope $c_j$ at $b_j$, so choosing the $c_j$ as the jumps of the slopes gives exactly the interpolant. The sup error behaves like $C/N^2$ (the column error$\cdot N^2$ settles near $25$, below the classical bound $\frac18\max|f''|\approx\frac{22\pi^2}8\approx27$), the rate of linear interpolation. Density with an explicit rate, and no training. (b) Gradient descent finds good tanh networks once the width is large enough; small widths stay far from $f$. Stone–Weierstrass (and Cybenko, Hornik) only promise that *some* network of *some* width is within $\varepsilon$: neither how wide, nor that gradient descent finds it. Training results depend on the random initialisation; change the seed and the errors for small widths move a lot.

### P4.2

In [ ]:
import numpy as np
from scipy.stats import binom

def bernstein(f, n, x):
    k = np.arange(n + 1)
    return binom.pmf(k[None, :], n, x[:, None]) @ f(k / n)

xs = np.linspace(0, 1, 2001)
for name, f in [("|x-1/2|", lambda x: np.abs(x - 0.5)), ("sin(2 pi x)", lambda x: np.sin(2 * np.pi * x))]:
    for n in [10, 100, 1000]:
        err = np.max(np.abs(bernstein(f, n, xs) - f(xs)))
        print(f"{name:12s} n={n:5d}: sup error {err:.2e}   err*sqrt(n)={err*np.sqrt(n):.3f}   err*n={err*n:.3f}")

For $|x-\frac12|$ the error is largest at the kink and decays like $n^{-1/2}$ ($\text{err}\cdot\sqrt n\approx0.399$). At $x=\frac12$ the error is $\mathbb E|\frac{S_n}n-\frac12|\approx\frac1{2\sqrt n}\mathbb E|Z|=\frac1{\sqrt{2\pi n}}$ for a standard Gaussian $Z$, by the central limit theorem of Volume 3, and $\frac1{\sqrt{2\pi}}\approx0.399$. For the smooth $\sin(2\pi x)$ it decays like $\frac1n$: by Taylor, $B_nf(x)-f(x)\approx\frac{x(1-x)}{2n}f''(x)$. Bernstein polynomials are slow, but they are positive operators and preserve monotonicity and convexity, which is why Bézier curves in every drawing program are built on them.

### P4.3

In [ ]:
import numpy as np
t = np.linspace(-1, 1, 20001); p = np.zeros_like(t)
for n in range(1, 201):
    p = p + 0.5 * (t ** 2 - p ** 2)
    if n in (1, 5, 20, 100, 200):
        e = np.abs(t) - p
        print(f"n={n:3d}: max error {e.max():.4f} at |t|={abs(t[e.argmax()]):.4f}   bound 2/(en)={2/(np.e*n):.4f}   min error {e.min():.1e}")

The error is never negative, stays below $\frac2{en}$, and is largest near $|t|\approx\frac{1.3}n$, close to where the proof of Lemma 4.18 located the maximum $\frac2n$ of its upper bound $se^{-ns/2}$; the true error is about $75\%$ of the bound. Away from $0$ the iteration converges geometrically; the kink at $0$ is what costs a factor $\frac1n$. The polynomial $p_n$ has degree $2^n$, absurdly high: the lemma is about existence, not efficiency (Newman found rational approximations of $|t|$ with error $e^{-c\sqrt n}$, far better than any polynomial of degree $n$ can do).

### P4.4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
D = 20

def ternary_digits(x):
    d = []
    for _ in range(D):
        x *= 3; k = min(int(x), 2); d.append(k); x -= k
    return d

def psi_digits(a):          # a: binary digits of a point of C
    u = sum(a[2 * k] * 2.0 ** -(k + 1) for k in range(D // 2))
    v = sum(a[2 * k + 1] * 2.0 ** -(k + 1) for k in range(D // 2))
    return np.array([u, v])

def Psi(x):
    d = ternary_digits(x)
    if 1 not in d:                         # x in C (to D digits)
        return psi_digits([k // 2 for k in d])
    i = d.index(1)                         # x lies in a gap removed at stage i+1
    alpha = sum(dk * 3.0 ** -(k + 1) for k, dk in enumerate(d[:i]))
    a_left = [k // 2 for k in d[:i]] + [0] + [1] * (D - i - 1)  # alpha + 3^-(i+1): left end of the gap = ...0222
    a_right = [k // 2 for k in d[:i]] + [1] + [0] * (D - i - 1) # right end = ...2000
    lo, hi = alpha + 3.0 ** -(i + 1), alpha + 2 * 3.0 ** -(i + 1)
    s = (x - lo) / (hi - lo)
    return (1 - s) * psi_digits(a_left) + s * psi_digits(a_right)

fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
for axis, m in zip(ax, [200, 2000, 40000]):
    P = np.array([Psi(x) for x in np.linspace(0, 1, m)])
    axis.plot(P[:, 0], P[:, 1], lw=0.4); axis.set_aspect("equal"); axis.set_title(f"{m} points")
plt.show()

rng = np.random.default_rng(2)
for h in [1e-2, 1e-3, 1e-4, 1e-5]:
    xs = rng.random(3000) * (1 - h)
    ratio = max(np.linalg.norm(Psi(x + h) - Psi(x)) for x in xs)
    print(f"h={h:.0e}: max |Psi(x+h)-Psi(x)| = {ratio:.2e},  / h = {ratio/h:.1e},  / h^(log2/log9) = {ratio/h**(np.log(2)/np.log(9)):.2f}")

The image fills the square, with visible structure: the curve spends the Cantor set filling the square and the gaps drawing straight segments. Moving $x$ by $h\approx3^{-2k}$ changes $2k$ ternary digits' worth of position, which moves both coordinates by about $2^{-k}$: so $|\Psi(x)-\Psi(y)|\lesssim|x-y|^{\alpha}$ with $\alpha=\frac{\log2}{\log9}\approx0.315$. The last column stays bounded while $|\Psi(x+h)-\Psi(x)|/h$ blows up. No curve onto the square can be Hölder with exponent more than $\frac12$: an $\alpha$-Hölder image of $[0,1]$ has dimension at most $\frac1\alpha$ (Chapter 11), and the square has dimension $2$. Hilbert's curve attains $\frac12$.

### P4.5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
f = lambda t, y: np.sqrt(min(abs(y), 1.0))

def euler(y0, n=10000):
    t = np.linspace(0, 1, n + 1); y = np.empty(n + 1); y[0] = y0
    for k in range(n):
        y[k + 1] = y[k] + f(t[k], y[k]) / n
    return t, y

for y0 in [0.0, 1e-12]:
    t, y = euler(y0)
    print(f"y0={y0:g}: y(1) = {y[-1]:.4f}   (t^2/4 at t=1 is 0.25)")

t = np.linspace(0, 1, 400)
for a in np.linspace(0, 0.8, 9):
    plt.plot(t, np.maximum(t - a, 0) ** 3, lw=1)
plt.title("solutions of y' = 3 y^(2/3), y(0) = 0"); plt.show()

From $y_0=0$ Euler's method stays at $0$ forever: it finds the zero solution. From $y_0=10^{-12}$ it escapes and ends at $y(1)\approx0.25$, along the solution $\frac{(t+c)^2}4$ with $c=2\sqrt{y_0}=2\times10^{-6}$: an invisible perturbation of the data selects a different solution. That is non-uniqueness made visible, and why Picard–Lindelöf's Lipschitz condition matters numerically, not only in theory. For $y'=3y^{2/3}$ the solutions $\max(t-a,0)^3$, $a\ge0$, all start at $0$: a whole family, and Arzelà–Ascoli's subsequences can reach any of them.

### P4.6

In [ ]:
import numpy as np, math
x = 1.0
for n in range(1, 10 ** 6 + 1):
    x = math.sin(x)
    if n in (10, 100, 10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6):
        print(f"n={n:8d}: x_n = {x:.6e}   x_n*sqrt(n/3) = {x*math.sqrt(n/3):.5f}")

$x_n\sqrt{n/3}\to1$ slowly (the correction is of order $\frac{\log n}n$). The reason is the expansion $\sin x=x-\frac{x^3}6+\cdots$: then $\frac1{x_{n+1}^2}-\frac1{x_n^2}\to\frac13$, so $\frac1{x_n^2}\approx\frac n3$. Ten correct digits of the fixed point $0$ would need about $3\times10^{20}$ iterations. Banach's theorem would promise geometric speed, but $\sin$ is not a contraction on $[0,1]$: $\frac{\sin x}x\to1$, so no single $k<1$ works. Edelstein's theorem gives convergence without any rate, and this example shows that no rate can be promised.